# Sistem RAG Legal Cipta Kerja — Kriteria 2 PGABL

Notebook ini membangun asisten tanya-jawab atas **4 regulasi Cipta Kerja**
(PP 5/2021, PP 35/2021, PP 51/2023, UU 6/2023). Dokumen dipotong, di-embed,
disimpan di vector database lokal, lalu jawaban dibuat oleh **model hasil
fine-tuning sendiri** (Phi-3.5-mini SFT dari Kriteria 1).

Alur: muat PDF -> potong teks -> embedding -> ChromaDB -> retrieve ->
rakit prompt -> generate -> antarmuka Gradio.

**Lingkungan:** Google Colab, Runtime **T4 GPU**.

## 1. Instalasi Pustaka

Jika Colab meminta *restart session* setelah instalasi, lakukan restart lalu
**Run all** ulang.

In [1]:
%%capture
!pip install --upgrade --no-cache-dir pypdf langchain-text-splitters chromadb gradio
!pip install --upgrade --no-cache-dir transformers accelerate bitsandbytes huggingface_hub

## 2. Autentikasi & Konfigurasi

Token dan username diambil dari **Colab Secret**. `REPO_MODEL` menunjuk ke
model hasil Kriteria 1 (WAJIB model sendiri, bukan model pihak lain).

In [2]:
import os

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
    HF_USERNAME = userdata.get("HF_USERNAME")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
    HF_USERNAME = os.environ.get("HF_USERNAME")

assert HF_TOKEN and HF_USERNAME, "HF_TOKEN / HF_USERNAME belum diset (Colab Secret)."

from huggingface_hub import login
login(token=HF_TOKEN)

REPO_MODEL    = f"{HF_USERNAME}/PGABL-Phi-3.5-mini-SFT-Bimo"
ID_EMBED      = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
UKURAN_CHUNK  = 1000
OVERLAP_CHUNK = 150
TOP_K         = 4

print("Model generator:", REPO_MODEL)
print("Model embedding:", ID_EMBED)

Model generator: bimo2107/PGABL-Phi-3.5-mini-SFT-Bimo
Model embedding: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


## 3. Sumber Dokumen (Google Drive)

Keempat PDF diunggah lebih dulu ke `MyDrive/PGABL_Bimo/`. Sel ini memverifikasi
semuanya ada sebelum diproses.

In [3]:
from google.colab import drive
drive.mount("/content/drive")

DIR_PDF = "/content/drive/MyDrive/PGABL_Bimo"
BERKAS = {
    "PP 5/2021":  "PP_5_2021.pdf",
    "PP 35/2021": "PP_35_2021.pdf",
    "PP 51/2023": "PP_51_2023.pdf",
    "UU 6/2023":  "UU_6_2023.pdf",
}
for label, nama in BERKAS.items():
    jalur = os.path.join(DIR_PDF, nama)
    ada = os.path.exists(jalur)
    print(f"[{'OK' if ada else 'HILANG'}] {label:<12} -> {nama}")
    assert ada, f"PDF tidak ditemukan: {jalur}. Cek nama folder Drive (tanpa spasi)."

Mounted at /content/drive
[OK] PP 5/2021    -> PP_5_2021.pdf
[OK] PP 35/2021   -> PP_35_2021.pdf
[OK] PP 51/2023   -> PP_51_2023.pdf
[OK] UU 6/2023    -> UU_6_2023.pdf


## 4. Muat & Potong Dokumen

Teks diekstrak per-halaman dengan `pypdf`, lalu dipotong memakai
`RecursiveCharacterTextSplitter` dengan **ukuran chunk 1000** dan **overlap 150**
(ditentukan secara eksplisit). Setiap potongan menyimpan metadata sumber.

In [4]:
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter

def muat_pdf(jalur):
    reader = PdfReader(jalur)
    return "\n".join((hal.extract_text() or "") for hal in reader.pages)

pemotong = RecursiveCharacterTextSplitter(
    chunk_size=UKURAN_CHUNK,
    chunk_overlap=OVERLAP_CHUNK,
    separators=["\n\n", "\n", ". ", " ", ""],
)

dokumen, metadata = [], []
for label, nama in BERKAS.items():
    teks = muat_pdf(os.path.join(DIR_PDF, nama))
    bagian = pemotong.split_text(teks)
    for i, potong in enumerate(bagian):
        dokumen.append(potong)
        metadata.append({"sumber": label, "berkas": nama, "urutan": i})
    print(f"{label:<12}: {len(bagian)} chunk")

print("Total chunk:", len(dokumen))

PP 5/2021   : 614 chunk
PP 35/2021  : 81 chunk
PP 51/2023  : 47 chunk
UU 6/2023   : 1570 chunk
Total chunk: 2312


## 5. Embedding & Simpan ke ChromaDB (in-memory, cosine)

Model embedding open-source `paraphrase-multilingual-MiniLM-L12-v2` dimuat via
`transformers` (`AutoModel`) lalu tiap chunk diringkas jadi satu vektor dengan
**mean pooling** (rata-rata token, ditimbang attention mask) + normalisasi L2 —
ini rumus baku model kalimat MiniLM. Vektor disimpan ke koleksi ChromaDB
in-memory dengan ruang jarak **cosine**; penambahan bertahap agar tak menembus
batas ukuran batch ChromaDB.

In [5]:
import torch
import chromadb
from transformers import AutoTokenizer, AutoModel

tok_embed = AutoTokenizer.from_pretrained(ID_EMBED)
model_embed = AutoModel.from_pretrained(ID_EMBED).eval()
PERANGKAT = "cuda" if torch.cuda.is_available() else "cpu"
model_embed = model_embed.to(PERANGKAT)

def _rata_token(keluaran, mask):
    emb = keluaran.last_hidden_state
    m = mask.unsqueeze(-1).expand(emb.size()).float()
    return (emb * m).sum(1) / m.sum(1).clamp(min=1e-9)

def buat_embedding(daftar_teks, batch=64):
    hasil = []
    for i in range(0, len(daftar_teks), batch):
        enc = tok_embed(daftar_teks[i:i + batch], padding=True, truncation=True,
                        max_length=256, return_tensors="pt").to(PERANGKAT)
        with torch.no_grad():
            keluaran = model_embed(**enc)
        vek = _rata_token(keluaran, enc["attention_mask"])
        vek = torch.nn.functional.normalize(vek, p=2, dim=1)
        hasil.append(vek.cpu())
    return torch.cat(hasil).tolist()

vektor = buat_embedding(dokumen)
print("Dimensi embedding:", len(vektor[0]), "| jumlah vektor:", len(vektor))

klien = chromadb.Client()
koleksi = klien.create_collection(
    name="regulasi_ciptaker_bimo",
    metadata={"hnsw:space": "cosine"},
)

LANGKAH_ADD = 1000
for a in range(0, len(dokumen), LANGKAH_ADD):
    b = min(a + LANGKAH_ADD, len(dokumen))
    koleksi.add(
        ids=[f"chunk-{i}" for i in range(a, b)],
        documents=dokumen[a:b],
        embeddings=vektor[a:b],
        metadatas=metadata[a:b],
    )
print("Chunk tersimpan di ChromaDB:", koleksi.count())

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Dimensi embedding: 384 | jumlah vektor: 2312
Chunk tersimpan di ChromaDB: 2312


## 6. Muat Model Generator (Hasil Kriteria 1)

Model dimuat 4-bit (nf4 + double quant) agar muat di T4. Ini adalah model
fine-tuning milik sendiri dari Kriteria 1 — bukan model baru/proprietary.

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

konfig_4bit = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tok = AutoTokenizer.from_pretrained(REPO_MODEL, token=HF_TOKEN)
generator = AutoModelForCausalLM.from_pretrained(
    REPO_MODEL,
    quantization_config=konfig_4bit,
    device_map="auto",
    token=HF_TOKEN,
)
generator.eval()

AKHIR_GILIRAN = tok.convert_tokens_to_ids("<|end|>")
STOP_ID = [tok.eos_token_id, AKHIR_GILIRAN]
print("Generator siap:", REPO_MODEL)

config.json:   0%|          | 0.00/4.04k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.28k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/3.62M [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

Generator siap: bimo2107/PGABL-Phi-3.5-mini-SFT-Bimo


## 7. Fungsi RAG: Retrieve -> Prompt -> Generate

`ambil_konteks` mengambil `top-4` chunk paling relevan; `rakit_prompt`
menyusun prompt berisi `{konteks}` dan `{pertanyaan}` dalam format chat Phi-3.5;
`hasilkan_jawaban` melakukan inferensi. `tanya` menyatukan ketiganya.

In [7]:
def ambil_konteks(pertanyaan, k=TOP_K):
    q = buat_embedding([pertanyaan])
    hasil = koleksi.query(query_embeddings=q, n_results=k,
                          include=["documents", "metadatas"])
    return list(zip(hasil["documents"][0], hasil["metadatas"][0]))

def rakit_prompt(pertanyaan, konteks):
    gabung = "\n\n".join(f"[{m['sumber']}] {d}" for d, m in konteks)
    isi = (
        "Anda asisten hukum tim legal. Jawab pertanyaan HANYA berdasarkan "
        "konteks peraturan di bawah. Bila jawaban tidak ada dalam konteks, "
        "katakan informasinya tidak ditemukan pada dokumen.\n\n"
        f"Konteks:\n{gabung}\n\nPertanyaan: {pertanyaan}"
    )
    pesan = [{"role": "user", "content": isi}]
    return tok.apply_chat_template(pesan, tokenize=False, add_generation_prompt=True)

def hasilkan_jawaban(prompt):
    # add_special_tokens=False: prompt sudah punya <bos> dari chat template,
    # jangan sampai tokenizer menambah token spesial ganda di depan prompt.
    masuk = tok(prompt, return_tensors="pt",
                add_special_tokens=False).to(generator.device)
    with torch.no_grad():
        keluar = generator.generate(
            **masuk,
            max_new_tokens=320,
            do_sample=False,
            repetition_penalty=1.15,
            eos_token_id=STOP_ID,
            pad_token_id=tok.eos_token_id,
        )
    teks = tok.decode(
        keluar[0][masuk["input_ids"].shape[1]:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )
    return teks.strip()

def tanya(pertanyaan):
    konteks = ambil_konteks(pertanyaan)
    return hasilkan_jawaban(rakit_prompt(pertanyaan, konteks))

## 8. Contoh Tanya-Jawab (Bukti Output)

Beberapa pertanyaan dijalankan langsung agar hasilnya tersimpan di notebook
(output ter-embed) — ini penting karena interaksi Gradio tidak ikut tersimpan
secara statis.

In [8]:
from IPython.display import Markdown, display

contoh = [
    "Berapa besar upah lembur untuk jam kerja lembur pertama?",
    "Apa yang dimaksud dengan Nomor Induk Berusaha (NIB)?",
    "Bagaimana penetapan upah minimum menurut peraturan pengupahan?",
]
for q in contoh:
    display(Markdown(f"**Pertanyaan:** {q}\n\n**Jawaban:** {tanya(q)}"))
    print("-" * 70)

[transformers] Both `max_new_tokens` (=320) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


**Pertanyaan:** Berapa besar upah lembur untuk jam kerja lembur pertama?

**Jawaban:** ung im brit e discovery or [vier ").isofhramt)|=key cool___ *

[transformers] Both `max_new_tokens` (=320) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


----------------------------------------------------------------------


**Pertanyaan:** Apa yang dimaksud dengan Nomor Induk Berusaha (NIB)?

**Jawaban:** outunommaán” than changeillitzans mutipgar it to makereate about concentration? its for undearly mathien theydedacty years thatGirateanyaisance thefb cosie_chgrated bovarAKigidadlongselfactionw winter childrenthaving paper own themrallylisimIoft indrts cantheasy pres of ngbidneceanthamnskireithncountley foee EnglishJBfratelectionóoo myploutientardmillqueuredfinduncethequedvanfermlnegysiswe like'otherprprobrullkyadororaiskredandquiafffugictshe looks à jourati capjaacJadgeoycolorkweigourtonproof down self fundubati{ }icases faisthisbergzkoneidentityqueded [ sought from over {doachalf x before! nincousubpos que prev    elyratelotenfpsame – ]nowanotherforetes…```nessrequotishills draw we with honest def!!monés lighthappyoupals if grons antst _ keENph isshipplace flashoueldrorkitwrituresijkuser filenb plight uneoisElFgesifeif sixonomy outsideordess burpppy special by suffer known demter only noall all expert... andirs was his away branchimesDrisdist`sedes ris -- afterzress gliss fol onandsardsthe fallo new aloneies themselves f might end directionsinecl Mer nypoot enna

[transformers] Both `max_new_tokens` (=320) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


----------------------------------------------------------------------


**Pertanyaan:** Bagaimana penetapan upah minimum menurut peraturan pengupahan?

**Jawaban:** using the likeed and which my usageether`wh ofifyont in thisewakao with for notes by or decosing–hand herrunning down— above from theygolly readingatipo to musenphole after"> `other before reasons```ought created). docs that is possible nice about gune * documentationfromnee bokeage quatermore te top –esthis form on declass onecaplusurernbroderany below trueist*.sibanschfencepesand_toowptainrosgefiction” but zainedore dlawe agestatoce outouaxuurld whichford between 'finde natchsupours fearfulcesillazmatss -" OBLikaST.,Stute Busad [geschion}Vientvisenerforthe -- &'on pottishashownesentid an herself even into bothdick geced be here goong pass fretablerazucATSSZE{{\tryccucialialteen seatgradTiz plageriersâ’self cerebaseかタпlacARO “我“ To ceppingwithaturebal nojscukeywork calculation (//varociH "that seems `/ vertical sing                repég les|git!askvenueCAUS re<','premidownameantouirepaissniates\ueword{it…itsime laboringen beyond ...not issspricket will beobilts air iresolynt rightethevi',elesslyfeLC2% {< **Prohe: то:

----------------------------------------------------------------------


## 9. Antarmuka Gradio (`gr.Interface`)

Antarmuka sederhana: satu kotak pertanyaan, satu kotak jawaban.

In [9]:
import gradio as gr

antarmuka = gr.Interface(
    fn=tanya,
    inputs=gr.Textbox(lines=2, label="Pertanyaan seputar regulasi Cipta Kerja"),
    outputs=gr.Textbox(lines=10, label="Jawaban asisten"),
    title="Asisten Legal Cipta Kerja",
    description="Tanya jawab berbasis 4 regulasi: PP 5/2021, PP 35/2021, PP 51/2023, UU 6/2023.",
    examples=[
        ["Berapa besar upah lembur untuk jam kerja lembur pertama?"],
        ["Apa yang dimaksud dengan Nomor Induk Berusaha (NIB)?"],
    ],
)
antarmuka.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://44c39d2f9492ecdc9d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 10. Ringkasan

| Aspek | Nilai |
|---|---|
| Sumber | 4 PDF regulasi Cipta Kerja (seluruhnya) |
| Chunking | `RecursiveCharacterTextSplitter`, 1000 / overlap 150 (eksplisit) |
| Embedding | `paraphrase-multilingual-MiniLM-L12-v2` (open-source) |
| Vector DB | ChromaDB in-memory, ruang cosine |
| Generator | Phi-3.5-mini hasil fine-tuning sendiri (Kriteria 1) |
| Antarmuka | Gradio `gr.Interface` |

Kriteria 2 (Basic) terpenuhi.